<font color=#009afa size=50>DAT-6002 - Fundamentals of Artificial Intelligence</font><br>
<font size=50>Semantic Networks</font><br>
<font size=20 color=#00c382>Medieval Weapons</font><br>

|**Assessment Code:** |DAT-6002|
|---|---|
|**Author:**|Tom Rowan|
|**Course:**|DAT6002_S1_26 *CHECK*|
|**Course Tutor:**|Imtiaz ****|


**Coding Syntax Note: _variable_name --> temporary / locally used variable, discarded after use.**


---

# Initialise Notebook

In [127]:
# Install Requirements Using Pip
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


### Import Python Libraries

In [128]:
# Standard Libraries
import sys
import time
import json
import random
from datetime import datetime
import re

# Requests
import requests

# Pandas and numpy
import pandas as pd
import numpy as np

# Matplot Lib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
%matplotlib inline

# Colorama
from colorama import Fore, Back, Style

# NetworkX
import networkx as nx

# PyVis
from pyvis.network import Network

# Do not print warnings!
import warnings
warnings.filterwarnings('ignore')

### Create a Timestamp

In [129]:
# Generate Timestamp
today = pd.Timestamp(datetime.today())
now_date_timestamp = str(today.strftime("%Y%m%d"))
now_datetime_timestamp = str(today.strftime("%Y%m%d-%H%M"))

### Functions for Pretty Output

print_output - Print coloured Output

In [130]:
# Function to print coloured output
def print_output (str1, str2, delim=':', colour='yellow'):
    if colour.upper() in dir(Fore):
        colour = colour.upper()
    else:
        colour = 'YELLOW'
    print (f'{getattr(Fore, colour)}{str(str1)}{delim}{Style.RESET_ALL} {str(str2)}')

### Check for Internet Access

In [131]:
_urls_to_check = ['http://connectivitycheck.gstatic.com/generate_204',
                  'https://www.github.com',
                  'https://datascience.daemonchild.com']

Set this to False if you have a local dataset

In [132]:
internet_required = False

In [133]:
# Function to Safely Check for Internet Connectivity
def check_internet (url_list):
    online = False
    for _url in url_list:
        try:
            response = requests.get(_url, timeout=3)
            if response.status_code == 200:
                online = True
                break
        except requests.RequestException:
            output_string = f"Check Internet failed for {_url}"
            print_output ('[FAIL]',output_string, colour='red', delim='')
    return online

# Perform the checks
if check_internet (_urls_to_check ):
    print_output ('[INTERNET]','Internet Connectivity Detected. Using Internet resources.',colour='green', delim='')
    online = True

else:
    print ()
    print_output ('[INTERNET]','No Internet Connectivity.', colour='red', delim='')
    online = False


# Stop running if Internet is required
if online == False & internet_required == True:
    sys.exit('Stopping notebook run here.')    


[INTERNET] Internet Connectivity Detected. Using Internet resources.


---
# Define Semantic Network Class

This class defines a semantic network for any data. The primary purpose is to allow simple gathering of data and to then visualise the network.

In [134]:
# Define Class SemanticNetwork
class SemanticNetwork:

    # Constructor
    def __init__(self, root_node="RootNode", structural_labels = ["IS-A", "TYPE-OF", "INSTANCE-OF"]):

        # Create graph object
        self.G = nx.DiGraph()
        self.root = root_node
        
        # Define palette (from https://coolors.co, using shades option)
        self.palette = {0:"#C3ADFF", 1:"#9670ff", 2:"#7847ff", 3:"#5A1FFF", 4:"#4100F5", 5:"#3100B8", 6:"#22007C"}
        self.bgcolour = "black"
        self.node_text_colour = "white"
        self.edge_label_colour = "#F0D90A"
        self.edge_colour_structural = "#c4830b81"
        self.edge_colour_property = "#7847ff81"

        self.structural_labels = structural_labels

    # Add structural relationships to the network (IS-A type)
    def add_relation(self, child, parent, label="IS-A"):
        self.G.add_edge(child, parent, label=label)

    # Add property relationships to the network (HAS-PART type)
    def add_property(self, node, value, label):
        self.G.add_edge(node, value, label=label)
    
    # Calculates distance from the root node
    # Used for dynamic scaling and chosing colours from palette
    def get_node_level(self, node):
        try:
            return nx.shortest_path_length(self.G.to_undirected(), source=node, target=self.root)
        except:
            return None

    # Export Network to json file 
    def export_to_json(self, filename="network.json"):
        #data = nx.edges(self.G)
        data = nx.node_link_data(self.G)    # deprecated...

        # *** Note: WRAP WITH A TRY 
        with open(filename, 'w') as f:
            json.dump(data, f, indent=4)

        print_output(f"Successfully saved", f"{len(self.G.nodes)} nodes to {filename}")

    # Import Network from json file 
    def import_from_json(self, filename="network.json"):
        
        # *** Note: WRAP WITH A TRY 
        with open(filename, 'r') as f:
            data = json.load(f)

        # Destructive
        self.G = nx.node_link_graph(data)
        print_output(f"Successfully loaded", f"{len(self.G.nodes)} nodes from {filename}")

    def print_levels (self):

        for s in self.G.nodes:
            print (s)
            spacer = {s: 0}
            for prereq, target in nx.dfs_edges(self.G, s):
                spacer[target] = spacer[prereq] + 2
                print (f'{spacer}+-{target}')
            print ('')


    # Create a dynamic visualisation of the network using PyVis, saved as HTML file
    def draw(self, filename="dynamic_network_diagram"):

        # Initiate the PyVis Network
        net = Network(height="1000", width="100%", bgcolor=self.bgcolour, font_color=self.node_text_colour, directed=True)
        
        dist = {n: self.get_node_level(n) for n in self.G.nodes() if self.get_node_level(n) is not None}
        
        print (dist)

        # Add Nodes 
        for node in self.G.nodes():
            level = dist.get(node, 99)
            level = self.get_node_level(node) or 99
            color = self.palette.get(level, "#D3D3D3")

            print (node, level)
            
            # Dynamic Scaling of nodes, but font doesn't seem to work...
            if level==0:
                dynamic_node_size = 120
                dynamic_font_size = 120
            else:
                dynamic_font_size = max(80 - (level * 8), 25) if level != 99 else 20
                dynamic_node_size = max(70 - (level * 7), 15) if level != 99 else 10

            
            net.add_node(
                node, 
                label=node, 
                color=color, 
                size = dynamic_node_size,
                font={'size': dynamic_font_size, 'face': 'arial', 'color': self.node_text_colour}
            )

        # Add Edges
        for source, target, data in self.G.edges(data=True):
            label = data.get('label')

            # Check if it is a structural label, as in an 'is-a' type relationship
            is_structural = label in self.structural_labels
            
            net.add_edge(
                source, 
                target, 
                label=label,
                width=5 if is_structural else 2,
                color= self.edge_colour_structural if is_structural else self.edge_colour_property,
                font={'color': self.edge_label_colour, 'size': 20, 'strokeWidth': 0}
            )

        # Define options for network
        options = {
            "nodes": {
                "font": {
                    "size": 30 
                }
            },
            "edges": {
                "font": {
                    "size": 18,
                    "strokeWidth": 0
                },
                "color": {"inherit": False}
            },
            "physics": {
                "enabled": True,
                "barnesHut": {
                    "gravitationalConstant": -13000, # Push nodes apart
                    "springLength": 300 # Lengthen the edges
                },
                "solver": "barnesHut"
            },
            "interaction": {
                "hideEdgesOnDrag": False,
                "hideEdgesOnZoom": False,
                "zoomView": True
            }
        }
        
        # Convert options for the visualisation JavaScript
        net.set_options(json.dumps(options))

        # Write out the diagram to the specified file
        net.write_html(filename)


    # The network can export itself to Prolog
    # Requires a rules file to be appended to the output
    def export_to_prolog (self, filename="prolog_file.pl",title = "Semantic Network",rules_filename='rules.pl'):

        # Separate storage for different types of facts
        is_a_facts = []
        has_a_facts = []
        # Empty File
        prolog_file = []

        for source, target, data in self.G.edges(data=True):
            label = data.get('label')

            src = f"{source.lower().replace(' ','_')}"
            tgt = f"{target.lower().replace(' ','_')}"
            label = label.lower().replace('-','_')
                
            # Grouping types together
            if label in ["is_a", "instance_of", "type_of"]:
                is_a_facts.append(f"is_a( {src}, {tgt} ).")
            else:
                # Special case when there is a 'from x to y' value
                if '-' in tgt:
                    tgt = f"{tgt.replace('-','_').replace('_',',')}"

                has_a_facts.append(f"has_a( {src}, {label}({tgt}) ).")

        # Sort the facts
        is_a_facts.sort()
        has_a_facts.sort()

        # Construct the file content in blocks
        header_block = [
            "%\n",
            f"% {title} Represented in Prolog\n",
            "%\n"
            f"% Auto generated from SemanticNetwork Class, by Tom Rowan",
            "%\n"
        ]
        
        prolog_file.append("\n% is_a facts\n")
        prolog_file.extend(is_a_facts)
        
        prolog_file.append("\n% has_a facts\n")
        prolog_file.extend(has_a_facts)

        # Append the rules
        with open(rules_filename, 'r') as f:
            rules_data = f.read()

        prolog_file.append(rules_data)

        # Save to file
        with open(filename, 'w') as f:
            for line in prolog_file:
                f.write(str(line)+'\n')
        
        print_output("Wrote File",f"Prolog written to {filename}.")

    # The network can be imported from a prolog file provided it follows the right structure
    def import_from_prolog (self, filename='prolog_file.pl'):

        # Empty lists
        is_a_list = []
        has_a_list = []

        # Regex patterns
        is_a_pattern = r"^is_a\(\s*(\w+),\s*(\w+)\s*\)\."
        has_a_pattern = r"^has_a\(\s*(\w+),\s*(\w+)\((.*?)\)\s*\)\."

        with open(filename, 'r') as file:
            data = file.read()

            # Extracting is_a facts
            is_a_list = re.findall(is_a_pattern, data, re.MULTILINE)
            
            # Extracting has_a facts
            has_a_list = re.findall(has_a_pattern, data, re.MULTILINE)

        # Add elements to the network
        for fact in is_a_list:
            #entity = fact[0].replace('_',' ').title()
            #parent = fact[1].replace('_',' ').title()
            entity = fact[0]
            parent = fact[1]
            self.G.add_edge(entity, parent, label='is-a')
    
        for fact in has_a_list:
            #entity = fact[0].replace('_',' ').title()
            #property = fact[1].replace('_',' ').title()
            #value = fact[2].replace('_',' ').title()
            entity = fact[0]
            property = fact[1]
            value = fact[2]
            self.G.add_edge(entity, value, label=property)

            #self.add_property(node=entity, value=value, label=property)

        print_output ("Added is_a facts",str(len(is_a_list)))
        print_output ("Added has_a facts",str(len(has_a_list)))


    def get_inherited_attribute(self, node, attribute_label):
        current = node
        while current:
            # Check current node for the specific property
            for _, neighbor, data in self.G.out_edges(current, data=True):
                if data.get('label') == attribute_label:
                    return neighbor
            # Climb up the hierarchy
            parents = [target for _, target, data in self.G.out_edges(current, data=True) 
                       if data.get('label') in self.structural_labels]
            current = parents[0] if parents else None
        return "N/A"


    def generate_report(self):
        report_data = []
        # Filter for 'Concept' nodes (those with a hierarchical level)
        concept_nodes = [n for n in self.G.nodes() if self.get_node_level(n) is not None]
        
        for node in concept_nodes:
            report_data.append({
                "Entity": node,
                "Level": self.get_node_level(node),
                "Damage": self.get_inherited_attribute(node, "causesDamage"),
                "Material": self.get_inherited_attribute(node, "primaryMaterial"),
                "Origin": self.get_inherited_attribute(node, "historicalOrigin"),
                "Era": self.get_inherited_attribute(node, "eraUsed")
            })
        return pd.DataFrame(report_data).sort_values("Level")


# Populate Network

In [135]:
# Create new network object (Level 0)
weapons_net = SemanticNetwork(root_node="Weapon")

# Add weapons to the hierarchy

### Level 1 - Categories
weapons_net.add_relation("Melee Weapon", "Weapon", label="TYPE-OF")
weapons_net.add_relation("Ranged Weapon", "Weapon", label="TYPE-OF")
weapons_net.add_relation("Siege Weapon", "Weapon", label="TYPE-OF")

### Level 2 - Categories
# Melee Weapon Categories
weapons_net.add_relation("Bladed Hand Weapon", "Melee Weapon", label="TYPE-OF")
weapons_net.add_relation("Blunt Hand Weapon", "Melee Weapon", label="TYPE-OF")
weapons_net.add_relation("Polearm", "Melee Weapon", label="TYPE-OF")

# Ranged Weapon Categories
weapons_net.add_relation("Francisca", "Ranged Weapon")
weapons_net.add_relation("Javelin", "Ranged Weapon")
weapons_net.add_relation("Bow", "Ranged Weapon")
weapons_net.add_relation("Arbalest", "Ranged Weapon")
weapons_net.add_relation("Gun", "Ranged Weapon")
weapons_net.add_relation("Artillery", "Ranged Weapon")

# Siege Weapon Categories
weapons_net.add_relation("Siege Tower", "Siege Weapon")
weapons_net.add_relation("Battering Ram", "Siege Weapon")
weapons_net.add_relation("Cat", "Siege Weapon")
weapons_net.add_relation("Weasel", "Siege Weapon")
weapons_net.add_relation("Chemical Weapon", "Siege Weapon")
weapons_net.add_relation("Biological Weapon", "Siege Weapon")
weapons_net.add_relation("Psychological Weapon", "Siege Weapon")


### Level 3 - Categories
# Bladed Hand Weapon Categories
weapons_net.add_relation("Sword", "Bladed Hand Weapon")
weapons_net.add_relation("Dagger", "Bladed Hand Weapon")
weapons_net.add_relation("Knife", "Bladed Hand Weapon")

weapons_net.add_property("Sword", "Steel", "primaryMaterial")
weapons_net.add_property("Sword", "Pommel", "hasPart")
weapons_net.add_property("Sword", "Hilt", "hasPart")
weapons_net.add_property("Sword", "Blade", "hasPart")
weapons_net.add_property("Sword", "Cut", "causesDamage")



# Blunt Melee Weapon Categories
weapons_net.add_relation("Club", "Blunt Hand Weapon")
weapons_net.add_relation("Mace", "Blunt Hand Weapon")
weapons_net.add_relation("Morning Star", "Blunt Hand Weapon")
weapons_net.add_relation("Holy Water Sprinkler", "Blunt Hand Weapon")
weapons_net.add_relation("Flail", "Blunt Hand Weapon")
weapons_net.add_relation("War Hammer", "Blunt Hand Weapon")
weapons_net.add_relation("Horsemans Pick", "Blunt Hand Weapon")

# Polearm Categories
weapons_net.add_relation("Quarterstaff", "Polearm")
weapons_net.add_relation("Spear", "Polearm")
weapons_net.add_relation("Winged Spear", "Polearm")
weapons_net.add_relation("Lance", "Polearm")
weapons_net.add_relation("Pike", "Polearm")
weapons_net.add_relation("Corseque", "Polearm")
weapons_net.add_relation("Fauchard", "Polearm")
weapons_net.add_relation("Glaive", "Polearm")
weapons_net.add_relation("Guisarme", "Polearm")
weapons_net.add_relation("Halberd", "Polearm")
weapons_net.add_relation("Danish Axe", "Polearm")
weapons_net.add_relation("Sparth", "Polearm")
weapons_net.add_relation("Bardiche", "Polearm")
weapons_net.add_relation("Pollaxe", "Polearm")
weapons_net.add_relation("Modern Flanged Mace", "Polearm")
weapons_net.add_relation("Maul", "Polearm")
weapons_net.add_relation("Becs de Corbin", "Polearm")

# Bow Categories
weapons_net.add_relation("Cross Bow", "Bow")
weapons_net.add_relation("Long Bow", "Bow")

# Guns and Artillery Categories
weapons_net.add_relation("Hand Cannon", "Gun")
weapons_net.add_relation("Arquebus", "Gun")

weapons_net.add_relation("Stone Thrower", "Artillery")
weapons_net.add_relation("Cannon", "Artillery")
weapons_net.add_property("Stone Thrower", "Pierrier", "alternateName")


### Level 4  (Weapons, and properties) and Level 5 (Specific Instances)
# Swords
weapons_net.add_relation("Arming Sword", "Sword")
weapons_net.add_relation("Broad Sword", "Sword")
weapons_net.add_relation("Falchion", "Sword")
weapons_net.add_relation("Long Sword", "Sword")

weapons_net.add_property("Arming Sword", "1000-1350", "eraUsed")
weapons_net.add_property("Arming Sword", "30-32", "lengthInches")
weapons_net.add_property("Arming Sword", "Light", "avgWeight")
weapons_net.add_property("Arming Sword", "Knight", "usedBy")
weapons_net.add_property("Arming Sword", "Two Edges", "hasEdges")

weapons_net.add_property("Broad Sword", "Thrust", "causesDamage")
weapons_net.add_property("Broad Sword", "1600-1799", "eraUsed")
weapons_net.add_property("Broad Sword", "30-32", "avgLengthInches")
weapons_net.add_property("Broad Sword", "Basket Hilt", "hasPart")
weapons_net.add_property("Broad Sword", "Substantial", "avgWeight")
weapons_net.add_property("Broad Sword", "Two Edges", "hasEdges")

weapons_net.add_property("Long Sword", "Thrust", "causesDamage")
weapons_net.add_property("Long Sword", "1350-1550", "eraUsed")
weapons_net.add_property("Long Sword", "40-48", "avgLengthInches")
weapons_net.add_property("Long Sword", "Knight", "usedBy")
weapons_net.add_property("Long Sword", "Europe", "historicalOrigin")
weapons_net.add_property("Long Sword", "1.2-2.4", "weightKg")
weapons_net.add_property("Long Sword", "Two Edges", "hasEdges")
weapons_net.add_property("Long Sword", "Hand-and-a-half sword", "alternateName")
weapons_net.add_property("Long Sword", "Great sword", "alternateName")
weapons_net.add_property("Long Sword", "Bastard sword", "alternateName")
weapons_net.add_property("Long Sword", "Fullers (Blood Grooves)", "hasPart")

weapons_net.add_relation("The Wallace Sword", "Long Sword", "INSTANCE-OF")
weapons_net.add_property("The Wallace Sword", "2.7", "weightKg") 
weapons_net.add_property("The Wallace Sword", "Scotland", "historicalOrigin")
weapons_net.add_property("The Wallace Sword", "Late 13th Century", "eraUsed")


weapons_net.add_property("Falchion", "1000-1500 AD", "eraUsed")
weapons_net.add_property("Falchion", "30-32", "lengthInches")
weapons_net.add_property("Falchion", "Mounted Knight", "usedBy")
weapons_net.add_property("Falchion", "One Edge", "hasEdges")
weapons_net.add_property("Falchion", "Europe", "historicalOrigin")

weapons_net.add_relation("Medici Falchion", "Falchion", label="INSTANCE-OF")
weapons_net.add_property("Medici Falchion", "Coat of Arms of Cosimo de Medici", "engravedWith")
weapons_net.add_property("Medici Falchion", "Gold plated", "secondaryMaterial")
weapons_net.add_property("Medici Falchion", "Florence", "historicalOrigin")
weapons_net.add_property("Medici Falchion", "1560", "usedEra")

weapons_net.add_relation("Cleaver Falchion", "Falchion", label="TYPE-OF")
weapons_net.add_relation("Cusped Falchion", "Falchion", label="TYPE-OF")


# Dagger and Knives
weapons_net.add_relation("Anelace", "Dagger") 
weapons_net.add_relation("Stiletto", "Dagger") 
weapons_net.add_relation("Poingnard", "Dagger") 
weapons_net.add_relation("Rondel", "Dagger") 

weapons_net.add_property("Knife", "Cut", "causesDamage")

weapons_net.add_property("Dagger", "Puncture", "causesDamage")
weapons_net.add_property("Dagger", "Two Edges", "hasEdges")
weapons_net.add_property("Dagger", "Knights", "usedBy")
weapons_net.add_property("Dagger", "Commoners", "usedBy")

weapons_net.add_property("Anelace", "20-30", "lengthInches") 

weapons_net.add_property("Stiletto", "Puncture", "causesDamage") 
weapons_net.add_property("Stiletto", "1100-1799", "eraUsed")
weapons_net.add_property("Stiletto", "No Edges", "hasEdges")    # True? It's just a point, isn't it?
weapons_net.add_property("Stiletto", "Misericorde", "alternateName")
weapons_net.add_property("Stiletto", "Germany and England", "historicalOrigin")


weapons_net.add_property("Poingnard", "Lightweight", "weight") 
weapons_net.add_property("Poingnard", "France", "historicalOrigin") 
weapons_net.add_property("Poingnard", "1100-1799", "eraUsed")

weapons_net.add_property("Rondel", "20", "lengthInches")



# Blunt Hand Weapons
weapons_net.add_property("Blunt Hand Weapon", "Bash", "causesDamage")
weapons_net.add_property("Blunt Hand Weapon", "Wood", "primaryMaterial")
weapons_net.add_property("Mace", "Puncture", "causesDamage")


# Artillery
weapons_net.add_relation("Traction Trebuchet","Stone Thrower")
weapons_net.add_relation("Counterweight Trebuchet","Stone Thrower")
weapons_net.add_relation("Onager","Stone Thrower")
weapons_net.add_relation("Mangonel","Stone Thrower")
weapons_net.add_relation("Traction Trebuchet","Stone Thrower")
weapons_net.add_relation("Ballista","Stone Thrower")
weapons_net.add_relation("Springald","Stone Thrower")

weapons_net.add_relation("Bombard","Cannon")
weapons_net.add_relation("Petard","Cannon")




# Generate Visualisations

In [136]:

# Create the interactive file
filename = "medieval_weapons_semantic_network_" + now_datetime_timestamp + ".html"
weapons_net.draw(filename)

{'Melee Weapon': 1, 'Weapon': 0, 'Ranged Weapon': 1, 'Siege Weapon': 1, 'Bladed Hand Weapon': 2, 'Blunt Hand Weapon': 2, 'Polearm': 2, 'Francisca': 2, 'Javelin': 2, 'Bow': 2, 'Arbalest': 2, 'Gun': 2, 'Artillery': 2, 'Siege Tower': 2, 'Battering Ram': 2, 'Cat': 2, 'Weasel': 2, 'Chemical Weapon': 2, 'Biological Weapon': 2, 'Psychological Weapon': 2, 'Sword': 3, 'Dagger': 3, 'Knife': 3, 'Steel': 4, 'Pommel': 4, 'Hilt': 4, 'Blade': 4, 'Cut': 4, 'Club': 3, 'Mace': 3, 'Morning Star': 3, 'Holy Water Sprinkler': 3, 'Flail': 3, 'War Hammer': 3, 'Horsemans Pick': 3, 'Quarterstaff': 3, 'Spear': 3, 'Winged Spear': 3, 'Lance': 3, 'Pike': 3, 'Corseque': 3, 'Fauchard': 3, 'Glaive': 3, 'Guisarme': 3, 'Halberd': 3, 'Danish Axe': 3, 'Sparth': 3, 'Bardiche': 3, 'Pollaxe': 3, 'Modern Flanged Mace': 3, 'Maul': 3, 'Becs de Corbin': 3, 'Cross Bow': 3, 'Long Bow': 3, 'Hand Cannon': 3, 'Arquebus': 3, 'Stone Thrower': 3, 'Cannon': 3, 'Pierrier': 4, 'Arming Sword': 4, 'Broad Sword': 4, 'Falchion': 4, 'Long Sword

In [137]:
# Save the network out
filename = "medieval_weapons_semantic_network_" + now_datetime_timestamp + ".json"
weapons_net.export_to_json(filename)

# Import from json
#weapons_net.import_from_json(filename)

Successfully saved: 119 nodes to medieval_weapons_semantic_network_20260223-1424.json


In [138]:
def json_to_prolog(json_filename="semantic_network.json", output_filename="semantic_network.pl",title = "Semantic Network",rules_filename='rules.pl'):
    with open(json_filename, 'r') as f:
        data = json.load(f)

    # Separate storage for different types of facts
    is_a_facts = []
    has_a_facts = []
    # Empty File
    prolog_file = []

    # 1. Categorize the links
    for edge in data.get('edges', []):
        src = f"{edge['source'].lower().replace(' ','_')}"
        tgt = f"{edge['target'].lower().replace(' ','_')}"
        label = edge['label'].lower().replace('-','_')

        # Grouping types together
        if label in ["is_a", "instance_of", "type_of"]:
            is_a_facts.append(f"is_a( {src}, {tgt} ).")
        else:
            # Special case when there is a 'from x to y' value
            if '-' in tgt:
                tgt = f"{tgt.replace('-','_').replace('_',',')}"

            has_a_facts.append(f"has_a( {src}, {label}({tgt}) ).")

    # Sort the facts
    is_a_facts.sort()
    has_a_facts.sort()

    # Construct the file content in blocks
    header_block = [
        "%\n",
        f"% {title} Represented in Prolog\n",
        "%\n"
        f"% Auto generated from SemanticNetwork Class, by Tom Rowan",
        "%\n"
    ]
    
    prolog_file.append("\n% is_a facts\n")
    prolog_file.extend(is_a_facts)
    
    prolog_file.append("\n% has_a facts\n")
    prolog_file.extend(has_a_facts)

    # Append the rules
    with open(rules_filename, 'r') as f:
        rules_data = f.read()

    prolog_file.append(rules_data)

    # Save to file
    with open(output_filename, 'w') as f:
        for line in prolog_file:
            f.write(str(line)+'\n')
    
    print_output("Wrote File",f"Prolog written to {output_filename}.")



In [139]:
#json_to_prolog(json_filename=filename, output_filename='2-Semantic-Network-Weapons.pl')

## Prolog to SN

In [140]:
# Load a semantic network from a prolog file
#
#   File must be formatted as follows:
#       is_a(mammal, animal)
#       is_a(canine, mammal)
#       ..
#       has_a(canine, makes_noise(bark))
#       has_a(canine, makes_noise(growl))
#       has_a(canine, legs(4))
#
def prolog_to_network(filename="semantic_network.pl", root_node="root_node"):

    semantic_net = SemanticNetwork(root_node=root_node)

    # Empty lists
    is_a_list = []
    has_a_list = []

    # Regex patterns
    is_a_pattern = r"^is_a\(\s*(\w+),\s*(\w+)\s*\)\."
    has_a_pattern = r"^has_a\(\s*(\w+),\s*(\w+)\((.*?)\)\s*\)\."

    with open(filename, 'r') as file:
        data = file.read()

        # Extracting is_a facts
        is_a_list = re.findall(is_a_pattern, data, re.MULTILINE)
        
        # Extracting has_a facts
        has_a_list = re.findall(has_a_pattern, data, re.MULTILINE)

    # Add elements to the network
    for fact in is_a_list:
        #entity = fact[0].replace('_',' ').title()
        #parent = fact[1].replace('_',' ').title()
        entity = fact[0]
        parent = fact[1]
        semantic_net.add_relation(entity, parent)

    for fact in has_a_list:
        #entity = fact[0].replace('_',' ').title()
        #property = fact[1].replace('_',' ').title()
        #value = fact[2].replace('_',' ').title()
        entity = fact[0]
        property = fact[1]
        value = fact[2]
        semantic_net.add_property(entity, value, property)

    print_output ("Is_a facts",str(len(is_a_list)))
    print_output ("Has_a facts",str(len(has_a_list)))

    return semantic_net

    
new_net = prolog_to_network(filename='2-Semantic-Network-Weapons.pl', root_node="weapon")
new_file = "new_semantic_network_" + now_datetime_timestamp + ".html"

new_net.draw(filename=new_file)

Is_a facts: 72
Has_a facts: 58
{'anelace': 4, 'dagger': 3, 'arbalest': 2, 'ranged_weapon': 1, 'arming_sword': 4, 'sword': 3, 'arquebus': 3, 'gun': 2, 'artillery': 2, 'ballista': 4, 'stone_thrower': 3, 'bardiche': 3, 'polearm': 2, 'battering_ram': 2, 'siege_weapon': 1, 'becs_de_corbin': 3, 'biological_weapon': 2, 'bladed_hand_weapon': 2, 'melee_weapon': 1, 'blunt_hand_weapon': 2, 'bombard': 4, 'cannon': 3, 'bow': 2, 'broad_sword': 4, 'cat': 2, 'chemical_weapon': 2, 'cleaver_falchion': 5, 'falchion': 4, 'club': 3, 'corseque': 3, 'counterweight_trebuchet': 4, 'cross_bow': 3, 'cusped_falchion': 5, 'danish_axe': 3, 'fauchard': 3, 'flail': 3, 'francisca': 2, 'glaive': 3, 'guisarme': 3, 'halberd': 3, 'hand_cannon': 3, 'holy_water_sprinkler': 3, 'horsemans_pick': 3, 'javelin': 2, 'knife': 3, 'lance': 3, 'long_bow': 3, 'long_sword': 4, 'mace': 3, 'mangonel': 4, 'maul': 3, 'medici_falchion': 5, 'weapon': 0, 'modern_flanged_mace': 3, 'morning_star': 3, 'onager': 4, 'petard': 4, 'pike': 3, 'poingn

In [141]:
sem_net = SemanticNetwork (root_node='Weapon')
sem_net.import_from_prolog(filename='2-Semantic-Network-Weapons-backup.pl')
sem_net.draw(filename='newnet.html')

Added is_a facts: 72
Added has_a facts: 58
{}
anelace 99
dagger 99
arbalest 99
ranged_weapon 99
arming_sword 99
sword 99
arquebus 99
gun 99
artillery 99
ballista 99
stone_thrower 99
bardiche 99
polearm 99
battering_ram 99
siege_weapon 99
becs_de_corbin 99
biological_weapon 99
bladed_hand_weapon 99
melee_weapon 99
blunt_hand_weapon 99
bombard 99
cannon 99
bow 99
broad_sword 99
cat 99
chemical_weapon 99
cleaver_falchion 99
falchion 99
club 99
corseque 99
counterweight_trebuchet 99
cross_bow 99
cusped_falchion 99
danish_axe 99
fauchard 99
flail 99
francisca 99
glaive 99
guisarme 99
halberd 99
hand_cannon 99
holy_water_sprinkler 99
horsemans_pick 99
javelin 99
knife 99
lance 99
long_bow 99
long_sword 99
mace 99
mangonel 99
maul 99
medici_falchion 99
weapon 99
modern_flanged_mace 99
morning_star 99
onager 99
petard 99
pike 99
poingnard 99
pollaxe 99
psychological_weapon 99
quarterstaff 99
rondel 99
siege_tower 99
sparth 99
spear 99
springald 99
stiletto 99
the_wallace_sword 99
traction_treb

In [142]:
print(sem_net.generate_report())


KeyError: 'Level'